# 🔍 Rows, columns and questions

**Time:** ~35 minutes &nbsp;•&nbsp; **Goal:** stop looking at the whole table and start **asking it things**. By the end you will be able to pull out exactly the rows you want, in one line, and you will know why half of those lines come with a pair of brackets you did not expect.

In the lecture you filtered with `df[df["Honeybees"] > 0]`. That works, and you will use it constantly. But it is one of three ways to select from a table, and it is worth knowing when each one is the right tool.

## Run this first

In [ ]:
# Run this first. Same file as notebook 1, with the header bug already fixed.
import urllib.request
from pathlib import Path

import pandas as pd

DATA_URL = "https://raw.githubusercontent.com/HocheggerLab/y3-bio-python/main/lab06/data/"

SURVEY = "pollinators.csv"
if not Path(SURVEY).exists():
    urllib.request.urlretrieve(DATA_URL + SURVEY, SURVEY)

df = pd.read_csv(SURVEY)
df.columns = df.columns.str.strip()

INSECTS = ["Beetles", "Hoverflies", "Flies", "Butterflies",
           "Honeybees", "Solitary_Bees", "Wasps", "Bumblebees"]

print(df.shape)
df.head(3)

## Skill 1 — `.loc` and `.iloc`

`df["Honeybees"]` gets you a column. To get **rows**, or rows *and* columns together, you need one of these two:

| | selects by | example |
|---|---|---|
| `.loc` | **label** — the index value and the column name | `df.loc[0, "Honeybees"]` |
| `.iloc` | **integer position** — like a list | `df.iloc[0, 13]` |

Right now the index is just `0, 1, 2, …`, so the two look almost the same. They stop looking the same the moment you sort or filter the table, because **the index travels with the row**. That is the point of it.

```python
df.loc[5]                       # the row labelled 5, as a Series
df.loc[5:8, ["Allotment", "Bumblebees"]]   # rows 5-8, two columns
df.iloc[:3, :4]                 # first 3 rows, first 4 columns
```

One gotcha worth meeting now: `.loc` slices are **inclusive** of the end label, unlike everything else in Python.

In [ ]:
# DEMO - label vs position
print("--- one row, by label ---")
print(df.loc[5, ["Date", "Allotment", "Crop_Generic", "Bumblebees"]])

print()
print("--- .loc is inclusive: 5 to 8 gives FOUR rows ---")
print(df.loc[5:8, ["Allotment", "Bumblebees"]])

print()
print("--- .iloc is exclusive, like a list slice: three rows ---")
print(df.iloc[5:8, :3])

In [ ]:
# DEMO - why the index matters
top = df.nlargest(3, "Bumblebees")
print(top[["Allotment", "Crop_Generic", "Bumblebees"]])

print()
print("iloc[0] -> the first row of the SORTED table:")
print(top.iloc[0][["Allotment", "Bumblebees"]].to_dict())

print()
print("the labels that came along for the ride:", top.index.tolist())

### 🔍 DIY challenge

1. Print rows 100 to 104 with `.loc`, showing only `Date`, `Crop_Generic` and `Honeybees`. How many rows did you get?
2. Get the same **shape** of answer with `.iloc`. What did you have to change?
3. `df.iloc[-1]` gives the last row. Is there a `.loc` equivalent? Try `df.loc[len(df) - 1]` and say why it happens to work here.

> 💡 If you find yourself counting columns to work out a number for `.iloc`, you wanted `.loc`.

In [ ]:
# TODO - your turn

## Skill 2 — A question, asked of every row

A comparison on a column gives you a Series of `True`/`False` — one per row. This is the **boolean mask** you met with numpy, and it does the same job here.

```python
df["Bumblebees"] > 0        # a Series of True/False, 1,620 long
df[df["Bumblebees"] > 0]    # the rows where it was True
```

Two things you get for free, exactly as with numpy:

- `mask.sum()` **counts** the `True`s
- `mask.mean()` gives you the **proportion** — because `True` is 1

In [ ]:
# DEMO - how much of this survey is zeros?
seen = df["Bumblebees"] > 0

print("watches with at least one bumblebee:", seen.sum(), "of", len(df))
print("as a proportion:", round(seen.mean(), 3))

print()
bb = df[seen]
print("when bumblebees WERE seen, the mean count was", round(bb["Bumblebees"].mean(), 2))
print("but across all watches it is only        ", round(df["Bumblebees"].mean(), 2))

That gap between the two numbers is not a detail — it is the shape of the data. Most watches saw nothing. Any mean you calculate later is a mean over a lot of zeros, and you will need to decide, every time, whether that is what you want.

### 🔍 DIY challenge

1. What proportion of watches recorded **no insects of any kind**? Build a row total first with `df[INSECTS].sum(axis=1)`.
2. How many watches had more than 100 flowers in view?
3. Of the watches that saw at least one honeybee, what was the largest count?

> 💡 Give your mask a name on its own line. `df[df[(df["a"] > 1) & (df["b"] < 2)]]` is where readable code goes to die.

In [ ]:
# TODO - your turn

## Skill 3 — Two conditions, and the brackets nobody warns you about

To combine conditions, pandas uses `&` (and), `|` (or), `~` (not). **Not** `and`, `or`, `not` — those are Python keywords that insist on a single `True`/`False`, and a mask is 1,620 of them.

And because `&` binds more tightly than `>`, **every condition needs its own brackets**:

```python
# right
df[(df["Bumblebees"] > 0) & (df["Year"] == 2018)]

# wrong - raises, or worse, quietly does something else
df[df["Bumblebees"] > 0 & df["Year"] == 2018]
```

This is the most common error in pandas, and you will make it today.

In [ ]:
# DEMO - bumblebees on runner beans, in 2018 only
mask = (df["Crop_Generic"] == "Rbean") & (df["Year"] == 2018) & (df["Bumblebees"] > 0)
print(mask.sum(), "matching watches")

df[mask][["Date", "Allotment", "Total_Flowers", "Bumblebees"]].head()

In [ ]:
# DEMO - what the wrong version actually does
try:
    df[df["Bumblebees"] > 0 & df["Year"] == 2018]
except Exception as e:
    print(type(e).__name__)
    print(e)

### 🔍 DIY challenge

1. Find the watches at `Weald` in `August` that saw more than two honeybees. How many are there?
2. Use `|` to find watches on **either** `Squash` **or** `Courgette`. How many?
3. Use `~` to find every watch that was **not** on a community plot, and count them. Check your answer against `df["Community"].value_counts()`.

> 💡 If a mask throws `The truth value of a Series is ambiguous`, you used `and` where you meant `&`.

In [ ]:
# TODO - your turn

## Skill 4 — `.isin()` and `.between()`

Once you are chaining three `|` conditions together, there is a shorter way.

```python
df[df["Crop_Generic"].isin(["Squash", "Courgette", "Cucumber"])]
df[df["Total_Flowers"].between(50, 100)]
```

`.isin()` takes a list and asks "is this row's value one of these?". `.between()` is inclusive at both ends. Both return an ordinary mask, so they combine with `&` and `|` like anything else.

In [ ]:
# DEMO - the cucurbits, in one readable line
cucurbits = ["Squash", "Courgette", "Cucumber"]
c = df[df["Crop_Generic"].isin(cucurbits)]

print(len(c), "watches on cucurbits")
print(c["Crop_Generic"].value_counts())

print()
mid = df[df["Total_Flowers"].between(50, 100)]
print(len(mid), "watches with 50-100 flowers in view")

### 🔍 DIY challenge

1. Pull out every watch from the three busiest allotments. Get the names from `.value_counts().head(3).index`.
2. Find the watches in the **summer months** — June, July and August — with `.isin()`.
3. Combine them: summer watches, at those three allotments, that saw at least one bumblebee. How many?

> 💡 `.value_counts()` returns a Series whose **index** is the names. `.index` hands that straight to `.isin()`.

In [ ]:
# TODO - your turn

## Skill 5 — Adding a column 🏆

Everything so far has taken things out of the table. Now put something in. Assigning to a name that does not exist yet **creates** the column:

```python
df["All_Insects"] = df[INSECTS].sum(axis=1)
```

This is where pandas earns its keep. The arithmetic runs over all 1,620 rows at once, and the result lines up row for row — no loop, no index bookkeeping.

The column worth building here is a **rate**: visits *per flower*. A watch that saw 5 bees on 10 flowers is a very different thing from one that saw 5 bees on 2,000 flowers, and the raw count cannot tell them apart.

In [ ]:
# DEMO - two new columns, and the difference between them
df["All_Insects"] = df[INSECTS].sum(axis=1)
df["Visits_Per_Flower"] = df["All_Insects"] / df["Total_Flowers"]

look = df.nlargest(5, "All_Insects")
print("--- busiest by raw count ---")
print(look[["Crop_Generic", "Total_Flowers", "All_Insects", "Visits_Per_Flower"]])

print()
print("--- busiest by visits per flower ---")
print(df.nlargest(5, "Visits_Per_Flower")[
    ["Crop_Generic", "Total_Flowers", "All_Insects", "Visits_Per_Flower"]])

Two completely different top-fives, from the same 1,620 rows. Neither is wrong. They answer different questions, and the job of an analyst is to know which one was asked.

We will come back to this properly in session 3 — it is the central idea of the whole lab.

### 🏆 DIY finale — build your own view of the data

1. Add a column `Bee_Visits` holding honeybees plus bumblebees plus solitary bees.
2. Add `Flowers_Per_Plant` — flowers divided by plants. Print its `.describe()`. Does anything look odd?
3. Build a filtered table `busy` containing only the watches that saw at least one insect **and** had at least 20 flowers in view. How many rows survive?
4. From `busy`, which crop appears most often? Which allotment?
5. Finally: is `Visits_Per_Flower` higher on community plots or individual plots? Compare the two means — and then keep the answer in mind, because in session 3 we will show you it is the wrong way to ask.

> 💡 Question 2 has a trap in it. `Plants` is the one column in this file with missing values, and dividing by a missing value gives you `NaN`. Notebook 3 is about exactly that.

In [ ]:
# TODO - your turn

## 🔭 Explore further

- The pandas indexing guide — long, but `.loc` vs `.iloc` is worth twenty minutes: <https://pandas.pydata.org/docs/user_guide/indexing.html>
- `df.query("Year == 2018 and Bumblebees > 0")` is a third way to filter, using a string. Try it. Some people love it; some find it hides the brackets they need to learn.

## Recap

You can now:

- select by label with `.loc` and by position with `.iloc`, and say which one you want
- build a boolean mask, count it with `.sum()` and take a proportion with `.mean()`
- combine conditions with `&`, `|`, `~` — each in its own brackets
- shorten a pile of `|` into `.isin()`, and a range into `.between()`
- create new columns, including the one that matters: a **rate** rather than a count

**Next:** `03_cleaning_basics` — the missing values, the duplicate rows, and the judgement calls no method can make for you.